In [ ]:
import sys
import os
import subprocess

# Check if we are running in Google Colab (remote Linux VM)
if 'google.colab' in sys.modules:
    print("Detected Google Colab environment. Setting up via GitHub...")

    # 1. Install required packages
    subprocess.run(["pip", "install", "-q", "torch_geometric", "prody"], check=True)

    # 2. Clone/update repository and submodules
    repo_url = "https://github.com/WSobo/Struct2Seq-GCN.git"
    target_dir = "/content/Struct2Seq-GCN"

    if not os.path.exists(target_dir):
        os.chdir('/content')
        print("Cloning repository (with submodules)...")
        subprocess.run(["git", "clone", "--recurse-submodules", repo_url], check=True)
    else:
        os.chdir(target_dir)
        print("Pulling latest changes...")
        subprocess.run(["git", "pull"], check=True)
        print("Updating submodules...")
        subprocess.run(["git", "submodule", "update", "--init", "--recursive"], check=True)

    # 3. Enter the project directory
    os.chdir(target_dir)
    print("Colab setup complete. Current Dir:", os.getcwd())

Detected Google Colab environment. Setting up via GitHub...
Pulling latest changes...
Colab setup complete. Current Dir: /content/Struct2Seq-GCN


# Struct2Seq-GCN: Pipeline Testing
This notebook runs the basic sanity checks for the inference and training pipelines of the Struct2Seq-GCN model.

In [15]:
import os
# Ensure we operate from the project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print("Current Working Directory:", os.getcwd())

Current Working Directory: /content/Struct2Seq-GCN


## Step 1: Test Inference Pipeline (Sanity Check)
Before training, make sure the model can successfully take a 3D structure, convert it to a graph, run it through the GCN, and output sequence predictions.

In [16]:
import sys
import os

# Set PYTHONPATH environment variable so python subprocesses can find 'utils' module natively
os.environ['PYTHONPATH'] = f"{os.environ.get('PYTHONPATH', '')}:{os.getcwd()}"

!python scripts/run.py --pdb LigandMPNN/outputs/autoregressive_score_wo_seq/1BC8_1.pt

Traceback (most recent call last):
  File "/content/Struct2Seq-GCN/utils/graph_builder.py", line 11, in <module>
    from data_utils import parse_PDB, featurize
ModuleNotFoundError: No module named 'data_utils'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/Struct2Seq-GCN/scripts/run.py", line 7, in <module>
    from utils.graph_builder import pdb_to_pyg_data
  File "/content/Struct2Seq-GCN/utils/graph_builder.py", line 15, in <module>
    from data_utils import parse_PDB, featurize
ModuleNotFoundError: No module named 'data_utils'


## Step 2: Test Training Loop
Confirm the data loader can successfully read the LigandMPNN `train.json` splits and start optimizing the weights for a couple of epochs.

In [ ]:
!python scripts/train.py \
    --json_train LigandMPNN/training/train.json \
    --json_valid LigandMPNN/training/valid.json \
    --pdb_dir LigandMPNN/inputs/ \
    --epochs 2